In [ ]:
#PyTorch  2.5.1 Python  3.12(ubuntu22.04)CUDA  12.4

In [1]:
pip install cupy-cuda12x 

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install gstools #用于生成高斯随机场

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.


In [3]:
import cupy as np
import gstools as gs
import matplotlib.pyplot as plt
import os
import sys

#有限差分法，得出训练集，由于步数太多，数据量很大，用numpy太慢，因此决定配置到GPU上跑。
# =============================================================================
# 1. GPU检查与参数定义
# =============================================================================

# 检查GPU是否可用，如果不可用则退出
if not np.cuda.is_available():
    print("错误：未找到可用的GPU设备。请确保CUDA环境已正确配置。")
    sys.exit()

print("GPU可用，正在使用CuPy进行加速...")

# --- 物理与计算参数 ---
Nx = 128            # x方向网格点数
Ny = 128            # y方向网格点数
Nt = 3001           # 时间步总数
x_range = 1.0       # x方向物理长度 (m)
y_range = 1.0       # y方向物理长度 (m)
t_range = 3.0       # 总模拟时长 (s)
no_initials =150   # 初始条件样本数
alpha = 0.01        # 热扩散系数 (m²/s)

# --- 生成坐标轴和离散步长 ---
x = np.linspace(0, x_range, Nx)
y = np.linspace(0, y_range, Ny)
t = np.linspace(0, t_range, Nt)

dx = x[1] - x[0]
dy = y[1] - y[0]
dt = t[1] - t[0]

# =============================================================================
# 2. 检查数值稳定性 (CFL条件)(显式有限差分法)
# =============================================================================
cfl_condition = alpha * (dt / dx**2 + dt / dy**2)
print(f"CFL条件值: {cfl_condition:.4f}")

if cfl_condition <= 0.5:
    print("CFL条件满足，数值解预期稳定。")
else:
    print("警告: CFL条件不满足，数值解可能会发散！")
    
# =============================================================================
# 3. 生成多样化的初始条件 (t=0)
# =============================================================================
# gstools不支持直接在GPU上生成，先在CPU上生成再转移
seed = gs.random.MasterRNG(42)#设置随机种子，确保实验可重复性

def generate_2d_initial(x_np, y_np):
    """使用 gstools 在CPU上生成单个高斯随机场作为初始条件。"""
    model = gs.Gaussian(dim=2, var=0.15, len_scale=2.0)#dim维度，var方差，len_scale控制协方差函数的平滑程度
    srf = gs.SRF(model, seed=seed())
    field = srf.structured([x_np, y_np])
    field_nonnegative = field - field.min()#确保是正数
    return field_nonnegative#非负

def apply_dirichlet_bc(U):
    """施加狄利克雷边界条件，强制边界值为0。"""
    U[0, :] = 0
    U[-1, :] = 0
    U[:, 0] = 0
    U[:, -1] = 0
    return U

U_np = np.zeros((Nx, Ny, no_initials), dtype=np.float32)#准备初始数据

for i in range(no_initials):
    # 在CPU上生成数据
    initial_field_cpu = generate_2d_initial(np.asnumpy(x), np.asnumpy(y))
    # 转移到GPU
    U_np[:, :, i] = np.array(initial_field_cpu, dtype=np.float32)
    U_np[:, :, i] = apply_dirichlet_bc(U_np[:, :, i])

# =============================================================================
# 4. 使用有限差分法进行时域求解
# =============================================================================
rx = alpha * dt / dx**2
ry = alpha * dt / dy**2

S = np.zeros((Nx, Ny, Nt, no_initials), dtype=np.float32)
S[:, :, 0, :] = U_np
#构建解集
for n in range(no_initials):
    for k in range(0, Nt-1):
        u = S[:, :, k, n]
        # 注意：这里的切片操作和计算全部在GPU上进行
        d2x = (u[:-2, 1:-1] - 2*u[1:-1, 1:-1] + u[2:, 1:-1])
        d2y = (u[1:-1, :-2] - 2*u[1:-1, 1:-1] + u[1:-1, 2:])
        S[1:-1, 1:-1, k+1, n] = u[1:-1, 1:-1] + rx * d2x + ry * d2y

# 应用边界条件
for k in range(1, Nt):
    for n in range(no_initials):
        S[:, :, k, n] = apply_dirichlet_bc(S[:, :, k, n])

# =============================================================================
# 5. 保存与降采样
# =============================================================================
num_time = 301
time_indices = np.linspace(0, Nt - 1, num_time, dtype=np.int32)
S_sampled = S[:, :, time_indices, :]
print(f"原始数据维度: {S.shape}")
print(f"经过时间降采样后，数据维度为: {S_sampled.shape}")

# 将数据从GPU转移到CPU进行保存
S_sampled_cpu = np.asnumpy(S_sampled)
np.save('results_128x128_gpu.npy', S_sampled_cpu)
print("128x128 结果已保存为 results_128x128_gpu.npy")

grid_sizes = [32, 64]
for size in grid_sizes:
    print(f"\n正在为 {size}x{size} 数据进行降采样...")
    x_indices = np.linspace(0, Nx - 1, size, dtype=int)
    y_indices = np.linspace(0, Ny - 1, size, dtype=int)
    smaller_S = S_sampled[np.ix_(x_indices, y_indices)]
    
    # 将数据从GPU转移到CPU进行保存
    smaller_S_cpu = np.asnumpy(smaller_S)
    np.save(f'results_{size}x{size}_gpu.npy', smaller_S_cpu)
    print(f"{size}x{size}结果已保存为 results_{size}x{size}_gpu.npy")


GPU可用，正在使用CuPy进行加速...
CFL条件值: 0.3226
CFL条件满足，数值解预期稳定。
原始数据维度: (128, 128, 3001, 150)
经过时间降采样后，数据维度为: (128, 128, 301, 150)
128x128 结果已保存为 results_128x128_gpu.npy

正在为 32x32 数据进行降采样...
32x32结果已保存为 results_32x32_gpu.npy

正在为 64x64 数据进行降采样...
64x64结果已保存为 results_64x64_gpu.npy
